# Reference · Day 7 studio — watching the penalty work

**Not a marking key.** You drew your own 300-house sample, so your numbers are not meant
to match these. What is worth comparing is the *shape*: which column was last standing,
whether your unscaled lasso noticed a change of units, and how few columns survived all
ten of your samples.

This uses sample `764`. Where the room's answers spread out, the spread is given too —
measured over thirty different samples, so you can see whether yours was typical. Every
result is followed by **What this shows**: what the numbers mean, and what they do not.

In [ ]:
# Find the course helpers (course/stat764.py): in your clone if this notebook sits inside
# it, or wherever STAT764_REPO points. Failing both, fetch the one file from GitHub.
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# A 300-house sample may contain no house with a pool; the imputer skips an
# all-missing column with a warning. Silence that one message only.
warnings.filterwarnings("ignore", message="Skipping features without any observed values")

ames = load("ames.csv")                      # all 2,930 houses, as on Tuesday
NUMERIC = [c for c in ames.select_dtypes("number").columns
           if c not in ("Order", "PID", "SalePrice")]      # 36: not the ids, not the answer


def numeric_pipeline(model, scale=True):
    """Impute, (optionally) scale, fit — the numeric half of the Meeting 2 skeleton."""
    steps = [("fill", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))          # learned from the training rows only
    return Pipeline(steps + [("model", model)])


def coefficients(model, X, y, scale=True):
    """Fit the pipeline, and hand back the model's coefficients: one per column of X."""
    return numeric_pipeline(model, scale).fit(X, y).named_steps["model"].coef_


sample = ames.sample(300, random_state=764)   # this reference's one sample of 300 houses
X_s, y_s = sample[NUMERIC], sample["SalePrice"]
ALPHAS = np.logspace(1, 5, 40)                # 40 penalties, 10 to 100,000, even on a log scale
print(f"{len(NUMERIC)} numeric columns, {len(sample)} houses")

## Question 1 — watch the coefficients move

Fit the lasso at every α and keep every coefficient. The x-axis **has to be on a log
scale**: α runs from 10 to 100,000, and on a linear axis everything interesting happens
in the first 1% of the plot.

In [ ]:
# One row per alpha, one column per predictor: each column of these tables is one line
# on the plot, a coefficient's path as the penalty rises.
lasso_paths = pd.DataFrame([coefficients(Lasso(alpha=a, max_iter=10_000), X_s, y_s)
                            for a in ALPHAS], index=ALPHAS, columns=NUMERIC)
ridge_paths = pd.DataFrame([coefficients(Ridge(alpha=a), X_s, y_s) for a in ALPHAS],
                           index=ALPHAS, columns=NUMERIC)

# Lasso on the left, ridge on the right, same y-axis so the two can be compared.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, paths, name in [(axes[0], lasso_paths, "lasso"), (axes[1], ridge_paths, "ridge")]:
    ax.plot(paths.index, paths.values, lw=1)
    ax.set_xscale("log")                                   # the one line that matters
    ax.axhline(0, color="black", lw=0.5)                   # zero, so you can see who reaches it
    ax.set_xlabel(f"{name} α (log scale)")
    ax.set_title(f"{name}: 36 coefficients as the penalty rises")
axes[0].set_ylabel("coefficient, $ per standard deviation")
plt.tight_layout()
plt.show()

Two different shapes from the same idea. The lasso's lines **hit zero and stay there**,
one after another. Ridge's lines bend toward zero together and never arrive. Put in
numbers:

In [ ]:
alive = lasso_paths.ne(0)                   # True wherever a lasso coefficient is not zero
# For each column that is ever in the model: the largest alpha at which it still is.
last = alive.loc[:, alive.any()].apply(lambda col: col[col].index.max())
print("  lasso — the largest α at which each column is still in the model:")
print(last.sort_values(ascending=False).head(5).round(0).to_string())
print(f"\n  lasso columns left at α = 3000:   "
      f"{(coefficients(Lasso(alpha=3000, max_iter=10_000), X_s, y_s) != 0).sum()} of 36")
print(f"  ridge coefficients exactly zero, anywhere on the grid:   "
      f"{int((ridge_paths == 0).sum().sum())}")

### What this shows

**Overall_Qual is the last one standing** — on this sample, and on every one of the
thirty samples we tried. Between 13 and 21 columns survive α = 3000, depending on the
sample (median 18).

**The lasso's zeros are real zeros; ridge has none.** That is the whole difference
between the two penalties: the lasso *removes* columns as the price of a coefficient
rises, and ridge only makes every coefficient smaller.

⚠ On a few samples one column **never varies** — `Pool_Area` when none of the 300 houses
has a pool. Ridge gives that column exactly zero, and so does everything else: a column
that is constant carries no information. That is not shrinkage, and it is not the lasso's
kind of zero. If your notebook said "ridge zeroed a coefficient", check whether this is
the one.

## Question 2 — change the units

Nothing about the houses changes. Only the units of one column.

In [ ]:
# Each row: one column, rescaled to new units, fit with the scaler on and then off.
# factor converts square feet into the new unit (1 acre = 43,560 sq ft; 1 sq ft = 0.0929 m²).
for column, unit, factor in [("Lot_Area", "square feet", 1),
                             ("Lot_Area", "acres", 1 / 43_560),
                             ("Gr_Liv_Area", "square feet", 1),
                             ("Gr_Liv_Area", "square meters", 0.092903)]:
    X_unit = X_s.copy()                     # a copy: X_s itself stays in square feet
    X_unit[column] = X_unit[column] * factor
    for scale in (True, False):
        coef = coefficients(Lasso(alpha=3000, max_iter=10_000), X_unit, y_s, scale)
        print(f"  {column:<12} in {unit:<14} scaler {'on ' if scale else 'off'}   "
              f"coefficient {coef[NUMERIC.index(column)]:>12.4f}")
    print()

### What this shows

With the scaler, neither change is visible: the coefficient is identical in either unit,
because the model only ever sees each column in standard deviations.

Without it, **the lasso throws away both columns** — on every one of thirty samples.
`Lot_Area` in acres needs a coefficient 43,560 times larger to mean the same thing, and
the lasso charges for size. `Gr_Liv_Area` is the more alarming one: it is the column
second most correlated with price, and measuring it in square meters was enough for an
unscaled lasso to decide it does not matter.

⚠ **The scaler goes inside the pipeline.** Scaling `X_s` once with
`StandardScaler().fit_transform` and then fitting the lasso gives the same coefficients
here — and it is still the Meeting 2 mistake, because the means and standard deviations
were estimated from rows the model will later be scored on.

## Question 3 — ten different samples

⚠ The loop has to draw a **new** sample each time. Reusing one sample ten times makes
every column "survive ten out of ten", which tells you nothing.

In [ ]:
kept = []
for s in range(10):
    draw = ames.sample(300, random_state=s)             # a NEW sample each time round
    kept.append(coefficients(Lasso(alpha=3000, max_iter=10_000),
                             draw[NUMERIC], draw["SalePrice"]) != 0)   # True = kept
kept = pd.DataFrame(kept, columns=NUMERIC)               # 10 samples x 36 columns
times = kept.sum().sort_values(ascending=False)          # per column: kept in how many?

print(f"  columns kept, sample by sample: {kept.sum(axis=1).tolist()}\n")
print(f"  kept in all ten        {(times == 10).sum():>3}   {times[times == 10].index.tolist()}")
print(f"  kept in some, not all  {((times > 0) & (times < 10)).sum():>3}")
print(f"  never kept             {(times == 0).sum():>3}")

fig, ax = plt.subplots(figsize=(9, 5.5))
times.sort_values().plot.barh(ax=ax, color="#4a6fa5")
ax.set_xlabel("samples (of 10) in which the lasso kept the column")
ax.set_title("Which columns the lasso keeps depends on which houses you drew")
plt.tight_layout()
plt.show()

### What this shows

**Seven of thirty-six columns survived all ten samples. Twenty-four came and went.** In
five different blocks of ten samples, the count kept every time was between 3 and 7.

On any *one* sample the lasso hands you a confident list of about eighteen columns, and
most of that list is a property of the sample rather than of houses. If you wrote down a
prediction before you ran this, the distance between your guess and this number is what
the studio was for.

⚠ **Now the tempting conclusion: "so the seven kept every time are the ones that matter,
and the five never kept do not."** Neither half follows.

- *Kept every time* means worth its penalty **at α = 3000, on 300 houses, with these
  other columns on offer**. Raise α and the list shrinks, all the way down to
  `Overall_Qual` alone. Stable is not the same as important.
- *Never kept* means the lasso never found the column worth paying for — which is not the
  same as the column telling you nothing about price. **Lab 4, Part 2** has you look hard
  at the rarely-kept columns and work out why.

That is Meeting 3 again: *which split you drew moves the answer more than which model you
chose.* This time the answer is a list of features instead of a score.

## The exit ticket — what you would say, and what you would refuse to say

| the sentence | could you defend it? |
|---|---|
| **"Overall quality is the strongest single predictor of price in these data"** | ✅ Yes. Last one standing on every one of thirty samples, and the column most correlated with price. |
| **"On this sample the lasso kept eighteen columns at α = 3000"** | ✅ True, and it says exactly how narrow it is. |
| **"At α = 3000, on 300-house samples, these seven columns were kept every time"** | ✅ True — and every qualifier in it is doing work. |
| **"The lasso found the eighteen features that drive price"** | ❌ Most of that list changes with the sample. |
| **"The seven columns kept every time are the ones that matter"** | ❌ Stable is not important. Change α, or the columns on offer, and the list changes. |
| **"A column the lasso never kept does not matter"** | ❌ Not worth its penalty here is not the same as no information about price. Lab 4, Part 2. |
| **"Lot area does not matter"** | ❌ An unscaled lasso said so because of the units. A scaled one kept it. |
| **"Ridge found that nothing is irrelevant"** | ❌ Ridge cannot zero a coefficient. It was never asked. |

## Check your own notebook against this

Not "did I get these numbers". These:

| # | the check |
|---|---|
| **1** | Is the x-axis of your path plot on a **log scale**? |
| **2** | Is the `StandardScaler` **inside the pipeline**, not applied to the data beforehand? |
| **3** | Did your ten samples actually **differ** — a new `random_state` each time? |
| **4** | Did you **write down a prediction** for Question 3 before you ran it — and is it still in the notebook? |
| **5** | Does the notebook survive **Restart & Run All**, top to bottom? A loop that reuses `X_s` or `sample` quietly changes every cell that runs after it. |

If 4 is a no, that is the one to fix. A guess you can check against the measurement is the
most useful thing a studio notebook can contain, and deleting it is the one edit that
costs you something.

⚠ **Lab 4 Part 2 asks the Question 3 question on the full design.** Measure it there
rather than quoting this.